In [1]:
account_path = '/content/drive/MyDrive/'

# Mount & Import

In [2]:
# mount the drive
from google.colab import drive
drive.mount('/content/drive/')
from google.colab import auth
auth.authenticate_user()

# load packages - basics
!pip install pynrrd
import os
import csv
import numpy as np
import pandas as pd
import math
import time
import json
import nrrd

Mounted at /content/drive/


In [3]:
# load packages - stats
!pip install scikit-posthocs
import scipy
from scipy import io
from scipy.fftpack import rfft, irfft, fftfreq
import scipy.stats as stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
import scikit_posthocs as scikit
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels

import random
from random import randrange, shuffle
import itertools
from itertools import combinations
import sklearn
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score

In [4]:
# load packages - plotting and template settings
import matplotlib as mpl
from matplotlib import pyplot as plt
%matplotlib inline

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [5]:
# load custom functions and sleap skeleton
%cd '/content/drive/MyDrive/Yu_escape_paper/Colab_github/src'
%run jy_utils.ipynb
sleap_nodes = ['le','re','fb','mb','rb','t1','t2','t3','tt'] # sleap_nodes_jy

/content/drive/MyDrive/Yu_escape_paper/Colab_github/src


# Experient list

In [6]:
# directories for the data and analysis
gsheet = 'https://docs.google.com/spreadsheets/d/1zJNHJJECC8FGB9S8tIh6QnoRihyoTZyKAgS-SE8ojwU/edit?gid=1421697431#gid=1421697431'
tab_name = 'tap_gsize'

experiment_class = 'Tap escape - group size'
path = '/content/drive/MyDrive/Yu_escape_paper/Behavior/' + experiment_class
data_path = path + '/Data/'
save_path = path + '/Analysis/'

metadata = read_metadata(gsheet,tab_name)
experiments = metadata.index

,experiment,fish_num,rig_mm,vid_px
0,jjm_20240223_s0,1,350,959
1,jjm_20240223_s1,1,350,959
2,jjm_20240223_s2,1,350,959
3,jjm_20240223_s3,1,350,959
4,jjm_20240226_s0_redo0223,1,350,947
...,...,...,...,...
70,jjm_20240313_s10,8,350,975
71,jjm_20240313_s11,8,350,975
72,jjm_20240313_s12,8,350,975
73,jjm_20240321-2_s3,8,350,968


# process

# lev 0

In [ ]:
# extract tracking from h5 & tapping from csv
# save out lev0.npz

for exp in experiments[::]:
  load_path = data_path + exp + '/'
  output_path = save_path + exp + '/'; os.makedirs(output_path, exist_ok=True)

  # metadata
  fish_num = metadata.fish_num[exp]
  rig_mm = metadata.rig_mm[exp]; vid_px = metadata.vid_px[exp]
  scale = rig_mm / vid_px # mm/pixel
  print('\nGroup:', exp, fish_num, 'fish')

  # -- step 1. Load h5 and csv -- #
  # timestamp.csv == absolute timestamps per video frame acquired
  # no header, one column
  time_sys = glob.glob(load_path + 'timestamp*.csv', recursive=True)[0]
  time_sys = np.array(pd.read_csv(time_sys, sep=',', header=0)).T[0]
  date = str(time_sys[0][0:10]); start_h = int(time_sys[0][11:13])
  start_m = int(time_sys[0][14:16]); start_s = float(time_sys[0][17:24])
  start_time = start_h*3600+start_m*60+start_s # time of the first frame

  time_sec = [(float(i[11:13])*3600 + float(i[14:16])*60 + float(i[17:24])) - start_time for i in time_sys]
  time_sec = np.array(time_sec)
  fps=float(len(time_sec)/max(time_sec))
  print(f"behavior length = {max(time_sec):.2f} sec, fps = {fps:.2f}")

  # tap_time.csv == start of digital output triggers tapper
  # no header, 2 columns: True/False, timestamp
  tapper = glob.glob(load_path + 'tap_time*.csv', recursive=True)[0]
  tapping = np.array(pd.read_csv(tapper,sep=',', header=0, skiprows=1)).T
  tap_time = tapping[1]
  tap_sec = [(float(i[11:13])*3600 + float(i[14:16])*60 + float(i[17:24])) - start_time for i in tap_time] # align to behavior time stamps
  tap_frames = find_nearest(time_sec,tap_sec) # align to behavior frames
  print(f"tap at #{tap_frames[0]} frame = {time_sec[tap_frames[0]]:.2f} sec")

  # load h5
  h5file = glob.glob(load_path + '*.h5', recursive=True)[0]
  h5file = h5py.File(h5file, 'r'); h5tracks = h5file['tracks'][:].T
  frame_num, _, _, track_num = h5tracks.shape
  print("frame_num x skeleton_size x 2 x fish_num", h5tracks.shape)

  # sanity check (1) consistent acquisition rate
  plt.figure(figsize=[4,1])
  plt.plot(time_sec);   plt.axvline(tap_frames[0],color='r')
  plt.title('time should be linear + tap (red)',fontsize=10); sns.despine(); plt.show()

  # -- step 2. Extract tracking from h5 -- #
  group_data = process_h5(h5tracks,fish_num,sleap_nodes, exp)

  # organize them into group arrays and save as lev0_basics.npz
  f_bodylength_px, f_bodylength_mm, f_nosex, f_nosey, f_x, f_y, f_tailx, f_taily,\
  f_heading, f_tail_angle, f_speed, f_ang_speed \
  = group_arrays(group_data,fish_num,frame_num,fps,scale)

  # SAVING
  np.savez(output_path + 'lev0_tap.npz', tap_frames=tap_frames)
  np.savez(output_path + 'lev0_basics.npz', fish_num=fish_num, fps=fps, frame_num=frame_num, \
            scale=scale, f_bodylength_px=f_bodylength_px, f_bodylength_mm=f_bodylength_mm,\
            f_nosex=f_nosex, f_nosey=f_nosey, f_x=f_x, f_y=f_y, f_tailx=f_tailx, f_taily=f_taily,\
            f_heading=f_heading, f_tail_angle=f_tail_angle, f_speed=f_speed, f_ang_speed=f_ang_speed)

  # sanity check (2) fish trajectory look clean
  fig, ax = plt.subplots(figsize=[3,3])
  ax.add_patch(plt.Circle((500,500),radius=500, edgecolor=(0,0,0,0.25), facecolor='none', linewidth=1))
  for f in range(fish_num): plt.plot(f_x[f][::20],f_y[f][::20],linewidth=.5,alpha=.75)
  plt.xlim(0,1000); plt.ylim(0,1000);
  for spine in ax.spines.values(): spine.set_visible(False)
  plt.show()

  # group level behaviors
  if fish_num == 1: continue
  ff_dist, ff_align, f_IID, f_IIA, f_closest_id, f_closest_dist, f_closest_align = analyze_neighbors(fish_num,f_heading,f_x,f_y,frame_num,scale,f_bodylength_px)
  group_area = 0 # default for 2-fish groups
  if fish_num > 2: group_area = fish_polygon_area2(f_x,f_y)

  # SAVING
  np.savez(output_path + 'lev0_group_basics.npz', group_area=group_area, ff_dist=ff_dist, \
          ff_align=ff_align, f_IID=f_IID, f_IIA=f_IIA, f_closest_id=f_closest_id,\
          f_closest_dist=f_closest_dist, f_closest_align=f_closest_align)

  # break